# Part 4: Python Memory Management & Internals

Python hides much of memory management, but senior Python developers should understand the ownership and execution model behind that convenience.

## Learning goals

By the end of this notebook, you should be able to:

- Explain heap objects, references, conceptual stack frames, allocation, and object lifetime.
- Describe CPython reference counting, cyclic garbage collection, generations, and the `gc` module.
- Diagnose common memory-retention problems and distinguish leaks from intentional caches or live references.
- Explain how CPython turns source code into bytecode and executes it in the Python virtual machine.
- Understand the role of `.pyc` files and `pymalloc` without needing to modify CPython itself.

> Important scope: these details describe CPython, the implementation used by most production Python installations. Other Python implementations may manage memory differently.

## 1. Python's memory model

Python programs manipulate **objects**, not raw memory addresses. Every object has a type, identity, value, and storage managed by the interpreter. Variables are names in namespaces that hold references to objects.

### Heap and stack, conceptually

- The **heap** is where dynamically allocated Python objects live. Lists, strings, instances, and dictionaries are heap objects.
- A **stack frame** is created for each active function call. It contains local-name bindings, evaluation state, and a link to enclosing/global scope. In CPython, frames and their locals are implemented as interpreter-managed objects; do not assume C-style stack layout.
- A local variable usually stores a reference to a heap object, not the object itself.

### Allocation and lifetime

Creating an object allocates storage; assigning a name usually only creates or changes a reference. In CPython, an object's reference count tracks active references. When the count reaches zero, CPython can usually deallocate it immediately. This is an implementation detail, not a language guarantee.

A reference can remain alive through a local variable, a container, a closure, a module global, a traceback, a cache, or a frame. A program that keeps an unwanted reference is retaining memory; this is often called a memory leak even when the allocator is working correctly.

### Reference counting and cyclic garbage

Reference counting cannot reclaim cycles such as `a -> b -> a`, because each object in the cycle still has a nonzero count even when the program can no longer reach the cycle. CPython's cyclic garbage collector supplements reference counting by finding unreachable groups of objects.

The collector is **generational**: newly created objects are checked more frequently, while long-lived objects are checked less often. Use `gc` to inspect thresholds, enable/disable automatic collection, force a collection, and inspect unreachable objects. Manual collection is a diagnostic or workload-specific tool, not a general substitute for releasing references.

In [1]:
import gc
import sys


class Tracked:
    def __del__(self):
        print("Tracked object finalized")


value = []
alias = value
print("same object:", value is alias)
print("references to list (includes temporary references):", sys.getrefcount(value))

del alias
print("after deleting alias, object still reachable:", value is not None)

tracked = Tracked()
print("tracked reference count:", sys.getrefcount(tracked))
del tracked
print("explicit reference removal completed")

print("GC thresholds:", gc.get_threshold())
print("GC counters before collection:", gc.get_count())
print("unreachable objects collected:", gc.collect())

# A cycle is not reclaimed by reference counting alone.
class Node:
    def __init__(self, name):
        self.name = name
        self.link = None

first = Node("first")
second = Node("second")
first.link = second
second.link = first
cycle_ids = (id(first), id(second))
del first, second
print("cycle created with object ids:", cycle_ids)
print("cyclic garbage collected:", gc.collect())

same object: True
references to list (includes temporary references): 3
after deleting alias, object still reachable: True
tracked reference count: 2
Tracked object finalized
explicit reference removal completed
GC thresholds: (2000, 10, 10)
GC counters before collection: (365, 8, 5)
unreachable objects collected: 67
cycle created with object ids: (1728789687536, 1728787717072)
cyclic garbage collected: 2


## 2. CPython internals

### What happens when Python executes code?

A simplified CPython pipeline is:

1. The tokenizer and parser read source text and build an internal representation.
2. The compiler transforms the code into a **code object** containing bytecode, constants, names, and metadata.
3. CPython may write the code object's serialized form to a `.pyc` file inside `__pycache__`.
4. The CPython interpreter loop, often called the **Python virtual machine**, executes bytecode instructions.
5. Instructions create and manipulate heap objects, references, frames, and exceptions.
6. Reference counting and cyclic GC manage object lifetime while allocators obtain and release memory.

Bytecode is an implementation detail and can change between Python versions. It is not native machine code. `dis` is useful for learning and diagnostics, but production code should depend on Python semantics rather than exact instruction names.

### `.pyc` files

A `.pyc` file stores compiled bytecode for an importable module. It speeds up later imports by avoiding repeated parsing and compilation when the cached bytecode is valid. It is a cache, not a complete executable and not a security boundary. Python checks metadata such as the source timestamp/size or a hash, depending on the invalidation mode.

### Reference counting, garbage collection, and `pymalloc`

CPython uses reference counting as its primary lifetime mechanism and a cyclic collector as a supplement. Below that layer, CPython's memory allocators manage raw memory. **`pymalloc`** is optimized for many small allocations and obtains larger arenas from the operating system, then serves smaller blocks from pools. Freeing a Python object does not necessarily return memory immediately to the OS; the allocator may retain it for reuse. This explains why process RSS can remain high after objects are deleted.

A high process memory reading does not automatically prove a leak. Investigate reachability, allocation traces, caches, fragmentation, and allocator reuse.

In [2]:
import dis
import importlib.util
import sys
import tempfile
from pathlib import Path


def calculate_total(first, second):
    total = first + second
    return total * 2


print("CPython implementation:", sys.implementation.name)
print("bytecode for calculate_total:")
dis.dis(calculate_total)

code = calculate_total.__code__
print("code object name:", code.co_name)
print("code constants:", code.co_consts)
print("code variable names:", code.co_varnames)

# Demonstrate where imported modules would cache compiled bytecode.
with tempfile.TemporaryDirectory() as directory:
    module_path = Path(directory) / "demo_module.py"
    module_path.write_text("answer = 42\n", encoding="utf-8")
    cache_path = Path(importlib.util.cache_from_source(str(module_path)))
    print("pyc path before import:", cache_path.exists())
    spec = importlib.util.spec_from_file_location("demo_module", module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    print("imported module value:", module.answer)
    print("pyc path after import:", cache_path.exists())
    del sys.modules[spec.name]

print("small integer size:", sys.getsizeof(0))
print("list container size:", sys.getsizeof([]))
print("list with one reference:", sys.getsizeof([None]))

CPython implementation: cpython
bytecode for calculate_total:
  8           RESUME                   0

  9           LOAD_FAST_LOAD_FAST      1 (first, second)
              BINARY_OP                0 (+)
              STORE_FAST               2 (total)

 10           LOAD_FAST                2 (total)
              LOAD_CONST               1 (2)
              BINARY_OP                5 (*)
              RETURN_VALUE
code object name: calculate_total
code constants: (None, 2)
code variable names: ('first', 'second', 'total')
pyc path before import: False
imported module value: 42
pyc path after import: True
small integer size: 28
list container size: 56
list with one reference: 64


## 3. Memory retention and leak diagnosis

A practical memory leak is memory that remains reachable or cannot be reused as expected after the operation that needed it has finished. Common causes include:

- unbounded module-level caches or registries;
- containers that are appended to but never pruned;
- callbacks and event handlers that retain objects;
- closures retaining large objects;
- long-lived threads, tasks, or queues;
- reference cycles involving objects with finalizers;
- tracebacks or debugging structures retaining frames.

Use ownership-oriented fixes: remove references, bound caches, unregister callbacks, close resources, clear queues, and use weak references where appropriate. `tracemalloc` compares allocation snapshots and points to Python source locations. It tracks Python allocations, not every native allocation, so pair it with process-level tools when needed.

Do not confuse memory usage with a leak. A live cache is retained by design; allocator arenas may remain available for future Python allocations; and temporary peaks may be normal. Measure before and after a repeated workload and inspect what remains reachable.

In [3]:
import gc
import tracemalloc


def allocate_records(count):
    return [{"index": index, "payload": "x" * 100} for index in range(count)]


tracemalloc.start()
before = tracemalloc.take_snapshot()
records = allocate_records(2_000)
after = tracemalloc.take_snapshot()

print("top Python allocation differences:")
for statistic in after.compare_to(before, "lineno")[:3]:
    print(statistic)

# Release the reference, then collect objects that are unreachable.
del records
gc.collect()
print("tracemalloc current and peak bytes:", tracemalloc.get_traced_memory())
tracemalloc.stop()

# A bounded cache avoids unbounded retention.
from functools import lru_cache


@lru_cache(maxsize=2)
def expensive(key):
    return key.upper()


for key in ("a", "b", "c"):
    expensive(key)
print("bounded cache statistics:", expensive.cache_info())
expensive.cache_clear()
print("cache after explicit cleanup:", expensive.cache_info())

top Python allocation differences:
C:\Users\kusol\AppData\Local\Temp\ipykernel_9312\1974756309.py:6: size=422 KiB (+422 KiB), count=5671 (+5671), average=76 B
C:\Program Files\Python313\Lib\codeop.py:117: size=1102 B (+744 B), count=14 (+11), average=79 B
C:\Program Files\Python313\Lib\tracemalloc.py:560: size=328 B (+328 B), count=1 (+1), average=328 B
tracemalloc current and peak bytes: (8277, 462049)
bounded cache statistics: CacheInfo(hits=0, misses=3, maxsize=2, currsize=2)
cache after explicit cleanup: CacheInfo(hits=0, misses=0, maxsize=2, currsize=0)


## 4. Practice lab

Attempt these exercises before consulting documentation or changing the examples above.

1. Create two names for the same list, mutate it through one name, then delete names one at a time. Explain which references keep the object alive.
2. Build a two-object reference cycle and use `gc.collect()` to observe collection. Repeat with `gc.disable()` and explain the difference in timing.
3. Use `gc.get_threshold()` and `gc.get_count()` before and after allocating many short-lived objects. Do not infer exact collection timing from one run.
4. Disassemble a function containing a loop, a conditional, and a function call. Identify loads, stores, jumps, and the return instruction.
5. Import a temporary module twice and inspect its `__pycache__` directory. Explain why a valid `.pyc` is a cache rather than source code.
6. Use `sys.getsizeof` to compare an empty list, a list with references, and the referenced objects themselves. Explain why `getsizeof` is shallow.
7. Create an intentionally unbounded cache, call it with many unique keys, then replace it with `lru_cache(maxsize=...)`. Compare the retention policy.
8. Use `tracemalloc` snapshots around a repeated workload and identify the top allocation site. Release references and explain what the snapshot does and does not prove.
9. Create an object with `__del__` that participates in a cycle. Investigate `gc.garbage` only if your Python configuration makes it relevant, and explain why explicit cleanup is preferable.
10. Write a short report answering: why can process memory remain high after `del` and `gc.collect()`? Include allocator reuse, fragmentation, and live references.

### Review checklist

- [ ] I distinguish a name/reference from the object it refers to.
- [ ] I understand heap objects and conceptual function stack frames.
- [ ] I know CPython reference counting does not solve cycles.
- [ ] I can explain generations and the role of the `gc` module.
- [ ] I can distinguish a real retention bug from allocator reuse or a bounded cache.
- [ ] I can use `tracemalloc` to locate Python allocation differences.
- [ ] I know bytecode is version-specific implementation detail.
- [ ] I understand what `.pyc` files cache.
- [ ] I know `pymalloc` may retain arenas instead of returning every freed block to the OS.

### Mental model

```text
source code
    -> parser/compiler
    -> code object and bytecode
    -> CPython virtual machine
    -> heap objects and references
    -> reference counting + cyclic GC
    -> CPython allocators such as pymalloc
```